# Cloud Kitchen P&L — Data Analysis Notebook
**Author:** Tathagata Ghosh  
**Python Version:** 3.14  
**Packages:** streamlit==1.45.1 | pandas==2.2.3 | plotly==5.24.1 | openpyxl==3.1.5  
**Live Dashboard:** https://kitchen-dashboard.streamlit.app  
**GitHub:** https://github.com/tathagat17/kitchen-dashboard

---
This notebook covers data preparation, exploration and key insights for the Cloud Kitchen P&L dashboard.  
Data: **344 stores | 5 cities | 4 zones | 6 months (Oct 2023 – Mar 2024)**

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Load data — same as app.py (header=1)
df = pd.read_excel('Untitled_spreadsheet.xlsx', header=1)
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())

## 2. Data Preparation (Matching final app.py exactly)

In [ ]:
# ── Computed columns — same as app.py ──
df['GM%']       = (df['GROSS MARGIN']   / df['NET REVENUE'] * 100).round(2)
df['CM%']       = (df['KITCHEN EBITDA'] / df['NET REVENUE'] * 100).round(2)
df['EBITDA%']   = (df['KITCHEN EBITDA'] / df['NET REVENUE'] * 100).round(2)
df['VARIANCE%'] = (df['VARIANCE']        / df['NET REVENUE'] * 100).round(4)

# ── Variance buckets — same as app.py ──
def variance_bucket(v):
    if v < 2:   return '(a) Var < 2%'
    elif v < 3: return '(b) Var 2% to 3%'
    elif v < 5: return '(c) Var 3% to 5%'
    else:       return '(d) Var > 5%'
df['VARIANCE BUCKET'] = df['VARIANCE%'].apply(variance_bucket)

# ── Revenue bands in lakhs — same as app.py ──
df['REVENUE BAND'] = pd.cut(
    df['NET REVENUE'] / 100000,
    bins=[0, 15, 25, 35, 45, float('inf')],
    labels=['(a) Below INR 15 lacs', '(b) INR 15 to 25 lacs',
            '(c) INR 25 to 35 lacs', '(d) INR 35 to 45 lacs',
            '(e) Above INR 45 lacs']
)

# ── Month ordering — same as app.py ──
month_order = ['Oct-2023','Nov-2023','Dec-2023','Jan-2024','Feb-2024','Mar-2024']
df['MONTH'] = pd.Categorical(df['MONTH'], categories=month_order, ordered=True)
df = df.sort_values('MONTH')

print('Processed Shape:', df.shape)
print('\nVariance Bucket Distribution:')
print(df['VARIANCE BUCKET'].value_counts())
print('\nRevenue Band Distribution:')
print(df['REVENUE BAND'].value_counts().sort_index())
print('\nEBITDA COHORT values:', df['EBITDA COHORT'].dropna().unique().tolist())
print('GM% range:', df['GM%'].min(), 'to', df['GM%'].max())

## 3. Dataset Overview

In [ ]:
print(f'Total Records     : {len(df)}')
print(f'Unique Stores     : {df["STORE"].nunique()}')
print(f'Cities            : {sorted(df["CITY"].unique().tolist())}')
print(f'Zones             : {df["ZONE MAPPING"].unique().tolist()}')
print(f'Avg Net Revenue   : ₹{df["NET REVENUE"].mean()/100000:.1f}L')
print(f'Avg EBITDA        : ₹{df["KITCHEN EBITDA"].mean()/100000:.1f}L')
print(f'Avg GM%           : {df["GM%"].mean():.1f}% (range: {df["GM%"].min():.1f}% to {df["GM%"].max():.1f}%)')
print(f'Avg CM%           : {df["CM%"].mean():.1f}% (range: {df["CM%"].min():.1f}% to {df["CM%"].max():.1f}%)')
print(f'EBITDA +ve        : {(df["EBITDA CATEGORY"]=="EBITDA +ve").mean()*100:.1f}%')
print(f'EBITDA -ve        : {(df["EBITDA CATEGORY"]=="EBITDA -ve").mean()*100:.1f}%')
print(f'EBITDA COHORT     : {df["EBITDA COHORT"].dropna().unique().tolist()}')
print(f'CM COHORT         : {df["CM COHORT"].dropna().unique().tolist()}')
print(f'REVENUE COHORT    : {df["REVENUE COHORT"].dropna().unique().tolist()}')
df[['NET REVENUE','GROSS MARGIN','KITCHEN EBITDA','GM%','CM%','VARIANCE%']].describe().round(2)

## 4. Dashboard 1 — Kitchen Level PNL

### Filters Used in app.py

The dashboard applies these filters (all from assignment requirements):
- **Dropdown filters:** Store, City, Zone, Month, Revenue Cohort, EBITDA Category, CM Cohort, EBITDA Cohort
- **Range sliders:** EBITDA Range (₹), CM% Range, Net Revenue Range (₹), GM% Range

In [ ]:
# Simulating Dashboard 1 filters — same logic as app.py
# Example: filter by city = Bangalore, EBITDA Category = EBITDA +ve

fdf = df.copy()

# Dropdown filters
sel_city          = 'Bangalore'      # Change to 'All' for no filter
sel_zone          = 'All'
sel_month         = 'All'
sel_store         = 'All'
sel_rev_cohort    = 'All'
sel_ebitda_cat    = 'EBITDA +ve'     # Change to 'All' for no filter
sel_cm_cohort     = 'All'
sel_ebitda_cohort = 'All'

# Range filters
sel_ebitda_range = (int(df['KITCHEN EBITDA'].min()), int(df['KITCHEN EBITDA'].max()))
sel_cm_range     = (int(df['CM%'].min()), int(df['CM%'].max()))
sel_rev_range    = (int(df['NET REVENUE'].min()), int(df['NET REVENUE'].max()))
sel_gm_range     = (int(df['GM%'].min()), int(df['GM%'].max()))

# Apply filters — same as app.py
if sel_city          != 'All': fdf = fdf[fdf['CITY']            == sel_city]
if sel_zone          != 'All': fdf = fdf[fdf['ZONE MAPPING']    == sel_zone]
if sel_month         != 'All': fdf = fdf[fdf['MONTH']           == sel_month]
if sel_store         != 'All': fdf = fdf[fdf['STORE']           == sel_store]
if sel_rev_cohort    != 'All': fdf = fdf[fdf['REVENUE COHORT']  == sel_rev_cohort]
if sel_ebitda_cat    != 'All': fdf = fdf[fdf['EBITDA CATEGORY'] == sel_ebitda_cat]
if sel_cm_cohort     != 'All': fdf = fdf[fdf['CM COHORT']       == sel_cm_cohort]
if sel_ebitda_cohort != 'All': fdf = fdf[fdf['EBITDA COHORT']   == sel_ebitda_cohort]

fdf = fdf[
    (fdf['KITCHEN EBITDA'] >= sel_ebitda_range[0]) & (fdf['KITCHEN EBITDA'] <= sel_ebitda_range[1]) &
    (fdf['CM%']            >= sel_cm_range[0])     & (fdf['CM%']            <= sel_cm_range[1])     &
    (fdf['NET REVENUE']    >= sel_rev_range[0])    & (fdf['NET REVENUE']    <= sel_rev_range[1])    &
    (fdf['GM%']            >= sel_gm_range[0])     & (fdf['GM%']            <= sel_gm_range[1])
]

print(f'Filtered: {fdf["STORE"].nunique()} stores | {len(fdf)} records')

In [ ]:
# Kitchen Snapshot Pivot Table — same as app.py Dashboard 1
pivot = fdf.pivot_table(
    index=['STORE','CITY','ZONE MAPPING'],
    columns='MONTH',
    values=['NET REVENUE','GM%','CM%','KITCHEN EBITDA','EBITDA%'],
    aggfunc='mean'
).round(2)
pivot.columns = [f'{c[0]} | {c[1]}' for c in pivot.columns]
pivot = pivot.reset_index()
print('Pivot shape:', pivot.shape)
pivot.head(3)

## 5. City-Level Performance

In [ ]:
city_perf = df.groupby('CITY').agg(
    Stores       =('STORE','nunique'),
    Avg_Rev_L    =('NET REVENUE', lambda x: round(x.mean()/100000,1)),
    Avg_EBITDA_L =('KITCHEN EBITDA', lambda x: round(x.mean()/100000,1)),
    Avg_GM_pct   =('GM%','mean'),
    Avg_CM_pct   =('CM%','mean'),
    Avg_Var_pct  =('VARIANCE%','mean')
).round(2).reset_index()
print(city_perf.to_string(index=False))

fig = px.bar(city_perf, x='CITY', y='Avg_EBITDA_L',
    color='Avg_EBITDA_L', color_continuous_scale='RdYlGn',
    title='Average EBITDA by City (₹ Lakhs)', text='Avg_EBITDA_L')
fig.update_layout(plot_bgcolor='white')
fig.show()

**Insight:** Ahmedabad leads EBITDA (₹7.2L). Mumbai lowest (₹6.3L) despite similar revenue.

## 6. Monthly Trend

In [ ]:
monthly = df.groupby('MONTH', observed=True).agg(
    Avg_Revenue=('NET REVENUE','mean'),
    Avg_EBITDA =('KITCHEN EBITDA','mean'),
    Avg_GM_pct =('GM%','mean'),
    Avg_CM_pct =('CM%','mean')
).round(2).reset_index()
print(monthly.to_string(index=False))

fig = make_subplots(rows=1, cols=2, subplot_titles=['Net Revenue Trend','EBITDA Trend'])
fig.add_trace(go.Scatter(x=monthly['MONTH'], y=monthly['Avg_Revenue'],
    mode='lines+markers', name='Revenue', line=dict(color='#3498db',width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=monthly['MONTH'], y=monthly['Avg_EBITDA'],
    mode='lines+markers', name='EBITDA', line=dict(color='#2ecc71',width=3)), row=1, col=2)
fig.update_layout(title='Monthly Revenue & EBITDA Trend', plot_bgcolor='white', height=400)
fig.show()

**Insight:** January 2024 is the peak month. November 2023 weakest — post-festive dip.

## 7. GM% and CM% Analysis

In [ ]:
# GM% ranges from 46.38% to 69.76% — used as slider in Dashboard 1
print('GM% Distribution:')
print(df['GM%'].describe().round(2))
print('\nCM% Distribution:')
print(df['CM%'].describe().round(2))
print('\nEBITDA COHORT Distribution:')
print(df['EBITDA COHORT'].value_counts())
print('\nCM COHORT Distribution:')
print(df['CM COHORT'].value_counts())

fig = make_subplots(rows=1, cols=2, subplot_titles=['GM% Distribution','CM% Distribution'])
fig.add_trace(go.Histogram(x=df['GM%'], name='GM%',
    marker_color='#3498db', nbinsx=20), row=1, col=1)
fig.add_trace(go.Histogram(x=df['CM%'], name='CM%',
    marker_color='#2ecc71', nbinsx=20), row=1, col=2)
fig.update_layout(title='GM% and CM% Distributions', plot_bgcolor='white', height=400)
fig.show()

**Insight:** GM% is tightly clustered between 46-70% (avg ~58%). CM% varies widely from -32% to 47% showing high variability in contribution margins across stores.

## 8. Profitable vs Loss-Making Stores

In [ ]:
print(df['EBITDA CATEGORY'].value_counts())
print(f'\n52.1% of store-months are LOSS MAKING')

pie_df = df['EBITDA CATEGORY'].value_counts().reset_index()
pie_df.columns = ['Category','Count']
fig = px.pie(pie_df, names='Category', values='Count', hole=0.4,
    color='Category',
    color_discrete_map={'EBITDA +ve':'#2ecc71','EBITDA -ve':'#e74c3c'},
    title='Profitable vs Loss-Making Stores')
fig.show()

## 9. Zone Performance

In [ ]:
zone_perf = df.groupby('ZONE MAPPING').agg(
    Avg_Rev_L    =('NET REVENUE', lambda x: round(x.mean()/100000,1)),
    Avg_EBITDA_L =('KITCHEN EBITDA', lambda x: round(x.mean()/100000,1)),
    Avg_GM_pct   =('GM%','mean'),
    Avg_CM_pct   =('CM%','mean')
).round(2).reset_index()
print(zone_perf.to_string(index=False))

fig = px.bar(zone_perf, x='ZONE MAPPING', y='Avg_CM_pct',
    color='Avg_CM_pct', color_continuous_scale='Blues',
    title='Average CM% by Zone', text='Avg_CM_pct')
fig.update_layout(plot_bgcolor='white')
fig.show()

**Insight:** East best (17.2% CM%), North worst (16.0%). 1.2% gap significant at scale.

## 10. Top & Bottom Stores

In [ ]:
store_perf = df.groupby('STORE').agg(
    Avg_Rev_L   =('NET REVENUE', lambda x: round(x.mean()/100000,1)),
    Avg_EBITDA_L=('KITCHEN EBITDA', lambda x: round(x.mean()/100000,1)),
    Avg_GM_pct  =('GM%','mean'),
    Avg_CM_pct  =('CM%','mean')
).round(2)
print('TOP 5 BY REVENUE:')
print(store_perf.sort_values('Avg_Rev_L', ascending=False).head(5))
print('\nBOTTOM 5 BY EBITDA:')
print(store_perf.sort_values('Avg_EBITDA_L').head(5))

top10 = store_perf.sort_values('Avg_EBITDA_L', ascending=False).head(10).reset_index()
fig = px.bar(top10, x='STORE', y='Avg_EBITDA_L',
    color='Avg_EBITDA_L', color_continuous_scale='Greens',
    title='Top 10 Stores by EBITDA', text='Avg_EBITDA_L')
fig.update_layout(plot_bgcolor='white', xaxis_tickangle=-30)
fig.show()

## 11. Dashboard 2 — Variance Level PNL

In [ ]:
# Sub-Dashboard 2a — Avg Variance % by Revenue Cohort
# Same logic as app.py Dashboard 2
rev_cohort_order = ['INR 20 to 30 lacs','INR 30 to 40 lacs','More than 40 lacs']

pivot2a = df.pivot_table(
    index='REVENUE COHORT', columns='MONTH',
    values='VARIANCE%', aggfunc='mean'
).round(4)
pivot2a = pivot2a.reindex([r for r in rev_cohort_order if r in pivot2a.index])
grand_row = pd.DataFrame(df.groupby('MONTH', observed=True)['VARIANCE%'].mean().round(4)).T
grand_row.index = ['Grand Total']
pivot2a = pd.concat([pivot2a, grand_row])
display2a = pivot2a.apply(lambda col: col.map(lambda x: f'{x:.2f}%' if pd.notnull(x) else '-'))
print('Sub-Dashboard 1 — Avg Variance % by Revenue Cohort:')
print(display2a.to_string())

In [ ]:
# Sub-Dashboard 2b — Store Count by Revenue Band
# Same logic as app.py Dashboard 2
rev_band_order = ['(a) Below INR 15 lacs','(b) INR 15 to 25 lacs',
    '(c) INR 25 to 35 lacs','(d) INR 35 to 45 lacs','(e) Above INR 45 lacs']

pivot2b = df.pivot_table(
    index='REVENUE BAND', columns='MONTH',
    values='STORE', aggfunc='count'
).fillna(0).astype(int)
pivot2b = pivot2b.reindex([r for r in rev_band_order if r in pivot2b.index])
grand_b = pd.DataFrame(pivot2b.sum()).T
grand_b.index = ['Grand Total']
pivot2b = pd.concat([pivot2b, grand_b])
print('Sub-Dashboard 2 — Store Count by Revenue Band:')
print(pivot2b.to_string())

## 12. Variance (Food Wastage) Heatmap

In [ ]:
print('Variance % by City:')
print(df.groupby('CITY')['VARIANCE%'].mean().round(4))
print('\nVariance % by Month:')
print(df.groupby('MONTH', observed=True)['VARIANCE%'].mean().round(4))

heat_df = df.pivot_table(index='CITY', columns='MONTH',
    values='VARIANCE%', aggfunc='mean').round(4)
fig = px.imshow(heat_df, color_continuous_scale='RdYlGn_r',
    text_auto='.2f', title='Food Wastage Heatmap — City × Month')
fig.show()

# Trend chart — same as Dashboard 2 bonus
chart2a = df.groupby(['MONTH','REVENUE COHORT'], observed=True)['VARIANCE%'].mean().reset_index()
fig2 = px.line(chart2a, x='MONTH', y='VARIANCE%', color='REVENUE COHORT',
    markers=True, title='Avg Variance % Trend by Revenue Cohort')
fig2.update_layout(plot_bgcolor='white')
fig2.show()

**Insight:** Pune highest wastage (0.63%), Hyderabad most efficient (0.61%). Pune Jan-2024 = 0.69% peak.

## 13. Performance Optimization — Caching Strategy

The dashboard `app.py` uses `@st.cache_data(ttl=300)` to optimize performance:

```python
@st.cache_data(ttl=300)  # Cache expires every 5 minutes
def load_data():
    df = pd.read_excel('Untitled_spreadsheet.xlsx', header=1)
    # All computed columns added here
    return df
```

**Benefits:**
- Data loaded from Excel **only once** per 5 minutes
- All filters/charts use in-memory cached dataframe — no repeated disk reads
- Manual **Refresh Data** button in sidebar clears cache instantly
- For real-time pipelines, replace `pd.read_excel()` with a live database query
- IST timezone used for accurate refresh timestamps

## 14. Key Insights Summary

| # | Insight | Severity |
|---|---------|----------|
| 1 | 52.1% stores EBITDA negative — majority loss-making | 🔴 Critical |
| 2 | Ahmedabad best city — highest EBITDA ₹7.2L, lowest wastage | 🟢 Positive |
| 3 | Mumbai underperforms — lowest EBITDA ₹6.3L vs similar revenue | 🔴 Action needed |
| 4 | January 2024 peak — highest revenue ₹35.6L and EBITDA ₹7.3L | 🟢 Seasonal |
| 5 | North zone lowest CM% 16.0% vs East zone 17.2% | 🟡 Investigate |
| 6 | Pune highest wastage 0.63% — Jan-2024 peak 0.69% | 🟡 Fix ops |
| 7 | Andrews Reed & Silva chronic loss — avg EBITDA -₹1.3L | 🔴 Immediate review |
| 8 | GM% tightly clustered 46-70%, CM% highly variable -32% to 47% | 🟡 Opportunity |
| 9 | All variance <1% but recovery possible at scale | 🟡 Opportunity |

### Recommendations
1. Investigate Mumbai for cost inefficiencies — similar revenue but lowest EBITDA
2. Replicate Ahmedabad best practices across network
3. North zone cost optimization — 1.2% CM gap significant at scale
4. Pune food wastage reduction program — target below 0.55%
5. Monthly review calls for all EBITDA -ve stores
6. Leverage January seasonal demand pattern across all cities
7. Stores with EBITDA Cohort '0% to 10%' need immediate intervention